In [6]:
import os
# In production the data lives in a PUBLIC Cloudflare R2 bucket (built and
# uploaded monthly). In local development we read the files written by
# data-prep/build_data.py.

R2_BASE = "https://pub-7559692ae8974bd6a6e7cc54db47a348.r2.dev"

# Set ENV=production in production, otherwise defaults to development.
IS_PROD = os.getenv("ENV") == "production"

# Equivalent of SvelteKit's `base`
BASE = ""

DATA_BASE = R2_BASE if IS_PROD else f"{BASE}/data"

In [9]:
import os

import boto3
R2_ENDPOINT_URL = "https://dd55a988a4598a3425259692fe6f7ba1.r2.cloudflarestorage.com/"

R2_ACCESS_KEY_ID = os.environ["R2_ACCESS_KEY_ID"]  # set in your shell; never commit keys
R2_SECRET_ACCESS_KEY = os.environ["R2_SECRET_ACCESS_KEY"]  # set in your shell; never commit keys
R2_BUCKET = os.environ.get("R2_BUCKET", "spl-checkouts-public")
R2_PREFIX = "authors/"

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT_URL,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
)

resp = s3.list_objects_v2(Bucket=R2_BUCKET)

for obj in resp.get("Contents", []):
    print(obj["Key"])

authors/0-/0-8syooogeki.json
authors/0-/0-paul-107.json
authors/0/0.json
authors/00/000-days-30.json
authors/00/000-homo-dj-s-1.json
authors/00/000-maniacs-10.json
authors/00/007-collective.json
authors/00/007-in-heaven.json
authors/00/007.json
authors/08/08hotel.json
authors/1-/1-4-5.json
authors/1-/1-arm-robbery.json
authors/1-/1-beatles-now.json
authors/1-/1-country-heroes.json
authors/1-/1-da-boy.json
authors/1-/1-garth-brooks-tribute-band.json
authors/1-/1-girl-nation.json
authors/1-/1-halloween-tv-movie-themes.json
authors/1-/1-hits-now.json
authors/1-/1-hits.json
authors/1-/1-holiday-carolers.json
authors/1-/1-holiday.json
authors/1-/1-karaoke-hits.json
authors/1-/1-katy-perry-tribute-band.json
authors/1-/1-speed-bike.json
authors/1-/1-western-holiday-all-stars.json
authors/10/10-32.json
authors/10/10-acre-wood.json
authors/10/10-feet-tokyo-ska-paradise-orchestra.json
authors/10/10-fuck-flicks.json
authors/10/10-indiecitos.json
authors/10/10-years.json
authors/10/10.json
authors

In [13]:
import json
import pandas as pd
rows = []

# List every object under authors/
paginator = s3.get_paginator("list_objects_v2")


for page in paginator.paginate(
    Bucket=R2_BUCKET,
    Prefix="authors/"
):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        print("Reading:", key)

        if not key.endswith(".json"):
            continue

        response = s3.get_object(
            Bucket=R2_BUCKET,
            Key=key
        )

        data = json.load(response["Body"])

        rows.append({
            "name": data.get("name"),
            "total": data.get("total"),
        })

df = pd.DataFrame(rows)

print(df.head())
print(len(df))

Reading: authors/0-/0-8syooogeki.json
Reading: authors/0-/0-paul-107.json
Reading: authors/0/0.json
Reading: authors/00/000-days-30.json
Reading: authors/00/000-homo-dj-s-1.json
Reading: authors/00/000-maniacs-10.json
Reading: authors/00/007-collective.json
Reading: authors/00/007-in-heaven.json
Reading: authors/00/007.json
Reading: authors/08/08hotel.json
Reading: authors/1-/1-4-5.json
Reading: authors/1-/1-arm-robbery.json
Reading: authors/1-/1-beatles-now.json
Reading: authors/1-/1-country-heroes.json
Reading: authors/1-/1-da-boy.json
Reading: authors/1-/1-garth-brooks-tribute-band.json
Reading: authors/1-/1-girl-nation.json
Reading: authors/1-/1-halloween-tv-movie-themes.json
Reading: authors/1-/1-hits-now.json
Reading: authors/1-/1-hits.json
Reading: authors/1-/1-holiday-carolers.json
Reading: authors/1-/1-holiday.json
Reading: authors/1-/1-karaoke-hits.json
Reading: authors/1-/1-katy-perry-tribute-band.json
Reading: authors/1-/1-speed-bike.json
Reading: authors/1-/1-western-holid

KeyboardInterrupt: 

In [34]:
import gzip
with gzip.open("search-index (1).json", "rt", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(
    data,
    columns=["slug", "name", "total"]
)

print(df.head())
print(df.shape)

                   slug                   name   total
0            mo-willems             mo willems  409417
1  james-patterson-1947  james patterson, 1947  387678
2              dr-seuss               dr seuss  296212
3     mary-pope-osborne      mary pope osborne  292198
4       dav-pilkey-1966       dav pilkey, 1966  274675
(253559, 3)


In [35]:
import pandas as pd

# number of authors
n_authors = len(df)

# first 10% (highest checkout totals)
top_10 = df.iloc[:int(n_authors * 0.10)]

# sample 200 authors
sample_top_10 = top_10.sample(
    n=200,
    random_state=41
)

print(sample_top_10.head())
print(sample_top_10.shape)

                     slug                name  total
14349         maki-enjoji         maki enjoji   1076
14568       brian-castner       brian castner   1057
12834    amy-hatvany-1972   amy hatvany, 1972   1223
12115  randa-abdel-fattah  randa abdel-fattah   1308
18656     kimiko-kajikawa     kimiko kajikawa    789
(200, 3)


In [36]:
sample_top_10.to_clipboard()

In [37]:
bottom_90 = df.iloc[int(n_authors * 0.10):int(n_authors * 0.40)]

sample_bottom_90 = bottom_90.sample(
    n=100,
    random_state=41
)

sample = pd.concat(
    [sample_top_10, sample_bottom_90],
    ignore_index=True
)

In [38]:
sample.to_clipboard()

In [43]:
import gzip
with gzip.open("title-index.json", "rt", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(
    data,
    columns=["title", "slug", "name", "total"]
)

print(df.head())
print(df.shape)

                                               title  \
0                                 Educated: A Memoir   
1  Braiding Sweetgrass: Indigenous Wisdom, Scient...   
2                            Where the Crawdads Sing   
3                                           Becoming   
4               All the Light We Cannot See: A Novel   

                       slug                       name  total  
0             tara-westover              tara westover  60279  
1  robin-wall-kimmerer-1953  robin wall kimmerer, 1953  51629  
2               delia-owens                delia owens  47499  
3       michelle-obama-1964       michelle obama, 1964  47415  
4        anthony-doerr-1973        anthony doerr, 1973  45329  
(26661, 4)


In [49]:
sample_titles = df.sample(
    n=300,
    random_state=42
 )

sample_titles.to_clipboard()